### Lab 9
#### Name: Vaibhav Morghade
#### Section: A-A3
#### Roll no.: 47
#### Date: 31-10-2026

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.cluster.hierarchy import fcluster

sns.set_theme(style='whitegrid')
plt.rcParams["figure.figsize"] = (8, 5)

df = pd.read_csv("Wholesale customers data.csv")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
sns.countplot(x="Channel", data=df)
plt.title("Customers per Channel")
plt.show()

Algorithm

In [ ]:
# Select the six spending features
features = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
X = df[features]

# Standardize the six spending features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
#Convert scaled values into a dataframe for easier inspection
X_scaled_df = pd.DataFrame(X_scaled, columns=features, index=X.index)

In [ ]:
print("First 5 scaled records:")
display(X_scaled_df.head())

In [ ]:
print("\nScaled feature means:")
print(X_scaled_df.mean().round(3))

In [ ]:
print("\nScaled feature standard deviations(approx. one):")
print(X_scaled_df.std(ddof=0).round(3))

In [ ]:
# Calculate WCSS (Within-cluster sum of squares)
wcss = []
k_values = range(1, 11)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

#plot
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), wcss, marker="o")
plt.title("Elbow Method for optimal number of cluster")
plt.xlabel("Number of clusters (k)")
plt.ylabel("WCSS/Inertia")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

In [ ]:
#Calculate silhouette scores to support the elbow detection
# (silhouette needs at least 2 clusters, so k starts from 2)
sil_k_values = range(2, 11)
silhouette_scores = []
for k in sil_k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

silhouette_table = pd.DataFrame({
    "Number of Clusters (k)": list(sil_k_values),
    "Silhouette Score": silhouette_scores
})

display(silhouette_table.round(4))

In [ ]:
suggested_k = int(
    silhouette_table.loc[silhouette_table["Silhouette Score"].idxmax(), "Number of Clusters (k)"]
)

print("K suggested by the highest silhouette score:", suggested_k)

In [ ]:
K = suggested_k

kmeans_model = KMeans(n_clusters=K, random_state=42, n_init=10)
kmeans_labels = kmeans_model.fit_predict(X_scaled)

In [ ]:
# add cluster label to a copy of the original dataset
df_kmeans = df.copy()
df_kmeans["KMeans_Cluster"] = kmeans_labels

print("KMeans clustering completed.")
print("\nNumber of customer in each cluster:")
print(df_kmeans["KMeans_Cluster"].value_counts().sort_index())

print("\nFirst 5 rows of the dataset with cluster labels:")
display(df_kmeans.head())

In [ ]:
#build the hierarchical clustering model
hierarchical_model = AgglomerativeClustering(n_clusters=K, metric="euclidean", linkage="ward")
hierarchical_labels = hierarchical_model.fit_predict(X_scaled)

In [ ]:
# store the cluster labels
df_hierarchical = df.copy()
df_hierarchical["Hierarchical_Cluster"] = hierarchical_labels

print("Hierarchical clustering completed.")
print("\nNumber of customer in each cluster:")
print(df_hierarchical["Hierarchical_Cluster"].value_counts().sort_index())

print("\nFirst 5 rows of the dataset with cluster labels:")
display(df_hierarchical.head())

In [ ]:
Z = linkage(X_scaled, method="ward", metric="euclidean")

plt.figure(figsize=(16, 7))
dendrogram(Z, leaf_rotation=90, leaf_font_size=6)

plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Customer index")
plt.ylabel("ward linkage Distance")
plt.tight_layout()
plt.show()

In [ ]:
#truncated dendrogram for easier interpretation
plt.figure(figsize=(16, 7))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=45, show_leaf_counts=True, leaf_font_size=10)

plt.title("Hierarchical Clustering Dendrogram(Truncated)-last 30 merged groups")
plt.xlabel("Customer or group")
plt.ylabel("ward linkage Distance")
plt.tight_layout()
plt.show()

In [ ]:
#reduce six scaled features to two principal components
pca = PCA(n_components=2)
X_PCA = pca.fit_transform(X_scaled)
print(
    "Variance explained by the 2 pca components:",
    round(pca.explained_variance_ratio_.sum() * 100, 2),
    "%"
)

#create a plotting dataframe
plot_df = pd.DataFrame({
    "PC1": X_PCA[:, 0],
    "PC2": X_PCA[:, 1],
    "KMeans_Cluster": kmeans_labels.astype(str),
    "Hierarchical_Cluster": hierarchical_labels.astype(str)
})

In [ ]:
#kmeans visualization
plt.figure(figsize=(9, 6))
sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="KMeans_Cluster", palette="tab10", s=70)

plt.title("Customer Segments-KMeans Clustering")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

In [ ]:
#function to calculate clustering evaluation metrics
def evaluate_clustering(X_data, labels, model_name):
    return {
        "Algorithm": model_name,
        "Number of Cluster": len(np.unique(labels)),
        "Silhouette Score": silhouette_score(X_data, labels),
    }

In [ ]:
#evaluate both models using the same scaled features
comparison = pd.DataFrame([
    evaluate_clustering(X_scaled, kmeans_labels, "KMeans"),
    evaluate_clustering(X_scaled, hierarchical_labels, "Hierarchical")
])

print("Comparison of clustering algorithms:")
display(comparison.round(4))